### Part 1.1: Precision DTypes: FP32 vs FP16 vs BF16

Each floating-point format uses a different bit allocation for the **sign**, **exponent**, and **mantissa** (fractional part):

| Format | Total Bits | Sign | Exponent | Mantissa | Range | Precision |
|--------|-----------|------|----------|----------|-------|-----------|
| **FP32** | 32 | 1 | 8 | 23 | ±3.4×10³⁸ | ~7 decimal digits |
| **FP16** | 16 | 1 | 5 | 10 | ±65,504 | ~3-4 decimal digits |
| **BF16** | 16 | 1 | 8 | 7 | ±3.4×10³⁸ | ~2-3 decimal digits |

### Key Trade-offs

**FP32 (Full Precision)**
- ✅ Highest precision for gradient accumulation
- ✅ Widest dynamic range
- ❌ Largest memory footprint (4 bytes/value)
- ❌ Slower compute on Tensor Cores

**FP16 (Half Precision)**
- ✅ 50% memory savings vs FP32
- ✅ 2-3× faster on modern GPUs
- ❌ Limited range (overflow/underflow issues)
- ❌ Smaller mantissa → precision loss
- ⚠️ Requires loss scaling to prevent gradient underflow

**BF16 (Brain Float)**
- ✅ 50% memory savings vs FP32
- ✅ Same exponent as FP32 → wider dynamic range
- ✅ Better for deep learning (avoids overflow/underflow)
- ❌ Smaller mantissa than FP16 → less precise
- ✅ No loss scaling needed (native support in recent GPUs)
- Supported on: NVIDIA A100+, newer TPUs






In [ ]:
# to find whether bf16 is supported in your GPU?
import torch

supported = torch.cuda.is_bf16_supported(including_emulation=False)
dtype16 = (torch.bfloat16 if supported else torch.float16)
print(f"Is native bfloat16 supported? {supported}")
dtype16

Is native bfloat16 supported? True


torch.bfloat16

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("=== LOADING STANDARD HF 4-BIT MODEL ===")

torch.cuda.empty_cache()

model_name = "facebook/opt-350m"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #quantization_config=quant_config,
    device_map='cuda:0',
    low_cpu_mem_usage=True,      # Helps reduce peak during load
    trust_remote_code=True
)

=== LOADING STANDARD HF 4-BIT MODEL ===


Loading weights: 100%|██████████| 388/388 [00:00<00:00, 846.47it/s]


In [9]:
from collections import Counter

def get_parm_dtypes(iterable, top_k=3):
    return Counter([p.dtype for p in iterable]).most_common(top_k)

In [10]:
print(model.get_memory_footprint()/1e6, get_parm_dtypes(model.parameters()))

662.392832 [(torch.float16, 388)]


By default, it should be all in FP32, but it's in FP16, that's because it is configured to be so in the model's config file.
https://huggingface.co/facebook/opt-350m/blob/main/config.json

### 1.1 Mixed Precision **Training**

On the one hand, 16-bit computing is faster than 32-bit computing. On the other hand, however, the loss in
precision gets compounded over time, operation after operation, thus leading to numerical issues. Perhaps we
can have our cake (32-bit) and eat it too (16-bit)?
Mixed precision is a powerful technique that balances **speed, memory efficiency, and numerical stability**:

### Core Concept

1. **Keep weights & inputs in full precision (FP32)**
   - Master weights stay in FP32 for gradient accumulation accuracy
   - Prevents gradient underflow and loss scaling issues

2. **Cast to half-precision before computation (FP16 or BF16)**
   - Forward pass uses FP16/BF16 for faster compute on GPUs (Tensor Cores)
   - Reduces memory footprint by ~50% compared to FP32

3. **Perform computation in half-precision**
   - Matrix multiplications, activations, etc. run in FP16/BF16
   - Modern GPUs (NVIDIA A100, RTX 3090) have specialized hardware for this

4. **Cast results back to FP32**
   - Accumulate gradients in FP32
   - Update weights in FP32 for stability

### Key Benefits

| Aspect | Benefit |
|--------|---------|
| **Speed** | 2-3× faster on modern GPUs with Tensor Cores |
| **Memory** | ~50% reduction in activation memory during forward/backward |
| **Convergence** | Maintains same accuracy as full FP32 with proper loss scaling |

### Why Both Precision Levels?

- **FP16 alone** → gradient underflow, numerical instability, loss divergence
- **FP32 alone** → slower, more memory, no hardware acceleration
- **Mixed (FP32 master + FP16 compute)** → best of both worlds

---

Torch internally uses autocast as a context manager to do operations with different datatypes for different steps. 

```python
with torch.autocast(device_type="cuda", dtype=torch.float16):
    res16 = mixed32(torch.randn(1000, 1000, dtype=torch.float32, device='cuda'))
res32 = res16.float()
```


Autocast should only wrap the forward pass and, perhaps, the loss computation. Backward passes under autocast are not recommended, so we’re finished with mixed precision at this point.

However, HuggingFace Trainer classes manage these operations internally and we can easily configure mixed-precision training by setting either the configuration’s fp16 or bf16 argument to True. This is indicative that 

### 1.2 Now let's read the model in bf16 since it is supported in our machine

In [60]:
model = AutoModelForCausalLM.from_pretrained(
"facebook/opt-350m", device_map='cuda:0', torch_dtype=dtype16
)
print(model.get_memory_footprint()/1e6, get_parm_dtypes(model.parameters()))

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 388/388 [00:00<00:00, 791.87it/s]


662.392832 [(torch.bfloat16, 388)]


### Part 2.1: Quantization

We can run the model faster with mixed precision, we can make it take lesser storage space also with 16 bit dtype (especially bf16 which even preserves performance) but that is still not enough since the models are getting huge everyday. Bits and Bytes like solutions provide us with the capability to go till 4-bit quant, enabling us to fit ever larger models in the same memory.

### 2.2 Understanding Quantization (BitsAndBytes)

### The Core Idea

Quantization reduces the precision of weights from high-bit formats (FP32, FP16) to lower-bit formats (8-bit, 4-bit integers) to save memory and accelerate computation.

**Simple example:**
```
Original FP32 weight: 0.123456789
Quantized to 8-bit:   42  (integer 0-255)
Back to FP16:         0.123456  (approximation, with loss)
```

### How BitsAndBytes Does It

BitsAndBytes uses **block-wise quantization** to keep accuracy reasonable:

1. **Divide weights into blocks** (default block_size=32 or 64)
   - Instead of one scale/zero-point for entire layer, each block has its own

2. **Find scaling factors per block:**
   ```
   scale = (max_value - min_value) / (2^bits - 1)
   zero_point = min_value
   ```

3. **Convert to integers:**
   ```
   qweight = round((weight - zero_point) / scale)
   ```
   Now qweight is 4-bit (0-15) or 8-bit (0-255)

4. **Store compactly:**
   - 4-bit: 2 values per byte → 524,288 values → 262,144 bytes
   - 8-bit: 1 value per byte → 524,288 values → 524,288 bytes
   - Plus scale & zero_point metadata per block

### During Inference (Dequantization)

```
output = (qweight * scale) + zero_point
```

This happens on-the-fly for each forward pass:
- Quantized weight (4-bit) stays in GPU memory
- Dequantized to higher precision for computation
- Inference slightly slower, but massive memory savings




When you do a forward pass with a 4-bit quantized model:

1. **Whole quantized model stays in memory (compressed)**
   - 4-bit weights + scales/zeros (compact storage)

2. **Layer-by-layer dequantization:**
   ```
   for each layer:
       qweight = load from GPU memory (4-bit)
       dequant_weights = (qweight * scale) + zero_point  # now 
       input = cast_to_computedtype(input)
       output = matmul(input, dequant_weights)
       output = cast_to_torchdtype(output) # e.g. bf16 or whatever was used in loading model so that it can be forwarded to the next layer which may or may not be a quantized layer
       discard dequant_weights  ← memory freed
   ```

3. **Only 1 layer's dequantized weights in memory at a time**
   - Not the whole model
   - This is why it works despite "can't fit full precision"




### Trade-offs

| Aspect | 4-bit | 8-bit |
|--------|-------|-------|
| **Memory** | ~8× compression | ~4× compression |
| **Speed** | Slower (dequant overhead) | Faster (byte-aligned) |
| **Accuracy** | More loss | Less loss |
| **Stability** | Needs careful tuning | More forgiving |



**Saves memory:** ✅ Only hold quantized weights + 1 layer dequantized at a time

**Costs speed:** ⚠️ Dequantization overhead per layer (~5-10% slower inference)

This is why BitsAndBytes has custom CUDA kernels—they make dequantization-on-the-fly fast enough to be practical.

---

### 2.3 Implementation
Now let's look at the 4-bit quantized model 

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — the actual 4-bit data type
    bnb_4bit_use_double_quant=True,     # Double quantization (extra memory saving)
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [63]:
model_4b = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map='cuda:0',
    low_cpu_mem_usage=True,      # Helps reduce peak during load
    trust_remote_code=True,
    torch_dtype=dtype16,
)

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 912.38it/s] 


In [64]:
print(model_4b.get_memory_footprint()/1e6, get_parm_dtypes(model_4b.parameters()))

207.835136 [(torch.bfloat16, 242), (torch.uint8, 146)]


Some layers are still in 16-bit, likely the output head and some normalization layers. The bulk of the model is in 4-bit though, which is where the memory savings come from.

### Now let's inspect the model a bit more closely

In [65]:
model_4b

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
      (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (

In [37]:
model_4b.model.decoder.project_out

Linear4bit(in_features=1024, out_features=512, bias=False)

In [36]:
model_4b.model.decoder.project_out.weight.data, model_4b.model.decoder.project_out.weight.shape

(tensor([[156],
         [ 78],
         [ 17],
         ...,
         [216],
         [169],
         [161]], device='cuda:0', dtype=torch.uint8),
 torch.Size([262144, 1]))

Now comparing it with the non-quantized version

In [38]:
model.model.decoder.project_out

Linear(in_features=1024, out_features=512, bias=False)

In [39]:
model.model.decoder.project_out.weight.data, model.model.decoder.project_out.weight.shape

(tensor([[ 0.0194,  0.0521, -0.0316,  ..., -0.0297, -0.1249,  0.0571],
         [ 0.0760,  0.1219, -0.0933,  ..., -0.0645, -0.0701, -0.0638],
         [ 0.0870, -0.0913,  0.1250,  ...,  0.0364, -0.0636, -0.1216],
         ...,
         [ 0.1248, -0.0332,  0.1119,  ..., -0.0177, -0.1246,  0.0665],
         [-0.0482,  0.0919, -0.0359,  ...,  0.0094, -0.1241, -0.0173],
         [ 0.0715,  0.1177,  0.1019,  ...,  0.0160,  0.0330, -0.0848]],
        device='cuda:0', dtype=torch.float16),
 torch.Size([512, 1024]))

We see that the torch.size property is different and dtype is also different for decoder.project_out layer

#### 2.3.1 Packing (a storage optimization for 4-bit or sub-byte quantization specifically)

why are the layers in 4bit model structured flat? It is not so for FP16?

`Linear4bit` is **BitsAndBytes' quantized linear layer implementation**. The logical shape is `in_features=1024, out_features=512`, which matches your FP16 weight shape of `[512, 1024]`.

But internally, `Linear4bit` stores the weights in a packed 4-bit 

The `.weight.data` shows the raw internal storage, not the logical weight matrix. The actual computation still uses the `[512, 1024]` semantics.



**The quantization flow:**

1. **Original FP16 weights:** 512 × 1024 = 524,288 FP16 values (2 bytes each)

2. **Quantize to 4-bit integers:** Each FP16 value is converted to a 4-bit integer (0-15), losing precision but saving space. You now have 524,288 4-bit integers.

3. **Pack those 4-bit integers:** 2 four-bit values fit in 1 byte → 524,288 / 2 = 262,144 bytes

4. **Store metadata separately:** BnB also stores `scale` and `zero_point` tensors (FP16 or less) per block of quantized values

**During forward pass:**
- The 4-bit integers are unpacked
- Dequantized using scales/zero-points to approximate the original FP16 values
- Used in the actual computation

So the 262,144 shape isn't "packing FP16 values"—it's packing the 4-bit **quantized integers**. The FP16 precision is already gone by then, converted to 4-bit representations.

That's why you store the scales/zeros: to recover reasonable approximations during inference without keeping the full-precision FP16 weights in memory.


---

8-bit also uses internal packing, but differently:

**4-bit packing:**
- 2 values per byte → `shape = [total_elements/2, 1]`
- More aggressive compression

**8-bit packing:**
- 1 value per byte → `shape = [total_elements, 1]` (or similar, less dramatic reshaping)
- Byte-aligned, so the "flattening" is less noticeable

Both use quantization codes (scales, zero-points) stored separately, so `.weight.data` won't show the true logical shape in either case.

If you load with `load_in_8bit=True`, you'd see `Linear8bitLt` (or similar) instead of `Linear4bit`, and the weight shapes would follow the same pattern—the stored tensor would be packed/flattened, but the layer still operates with its declared `in_features` and `out_features`.

Naive approach:
- Store each 4-bit integer as a full byte (wasteful)
- Shape stays `[512, 1024]`
- Uses 2× memory it could

BnB approach:
- Pack 2 4-bit ints per byte (efficient)
- Shape becomes `[262144, 1]`
- Uses minimal memory


---

### 2.4 4-bit Config Important Params


### 1. `load_in_4bit=True`
- **What it does**: Tells bitsandbytes to quantize the model weights to **4 bits per parameter** (instead of 16 or 32 bits).
- **Why we use it**: This is the whole point — massive memory reduction.  
  A 3B model in fp16 takes ~6 GB just for weights. In 4-bit → roughly **~1.5–2 GB** for weights → leaves headroom for activations, optimizer states, gradients (especially important during fine-tuning).
- **Alternatives**: `load_in_8bit=True` (worse compression), or leave it off (fp16/bf16 → OOM on your GPU for anything > ~2–3B).

### 2. `bnb_4bit_quant_type="nf4"`
This is the **single most impactful setting for quality**.

- **What it does**: Chooses the 4-bit **data type / representation** for the quantized weights.
- Two realistic options in bitsandbytes (2026):
  - `"fp4"`  = pure 4-bit floating point (1 sign + 2 exponent + 1 mantissa bits, standard IEEE-like float)
  - `"nf4"`  = **NormalFloat4** — a hand-crafted 4-bit float whose value bins are optimized assuming the **weights follow a roughly normal (Gaussian) distribution** (which is very common after initialization or after pre-training).

- **Why "nf4" instead of "fp4"**?
  - Neural network weights (especially after pre-training) are **not uniformly distributed** — they cluster around zero with long tails → normal-like.
  - `"nf4"` allocates more representable values near zero and in the typical range → **lower quantization error** for the same bit-width.
  - Empirical result (from QLoRA paper + hundreds of follow-up experiments): **nf4 gives noticeably better downstream performance** (perplexity, fine-tuning stability, final accuracy) than fp4 when doing 4-bit training / QLoRA.
  - Hugging Face blog & docs explicitly recommend: **"use NF4 for higher precision"**.

- **Rule of thumb** (still valid in 2026):
  - Use **nf4** when you care about quality (almost always when fine-tuning or doing serious inference).
  - Use **fp4** only if you have a strange model whose weights are **very non-Gaussian** (very rare).

### 3. `bnb_4bit_use_double_quant=True`
- **What it does** (also called **nested quantization**):
  - When quantizing to 4-bit, bitsandbytes computes scaling factors (one per block of weights, usually 64–128 values).
  - These scaling factors themselves are originally stored in fp32 → they take extra memory.
  - With `double_quant=True`, those scaling factors are **themselves quantized to 8-bit** → saves ~0.37–0.4 bits per parameter overall.

- **Why turn it on?**
  - **Memory saving is basically free**: ~0.4 bits/parameter → for a 3B model ≈ 150–200 MB saved, sometimes more.
  - **Quality degradation is very small** (often < 0.1% relative perplexity increase, frequently negligible).
  - Hugging Face rule of thumb: **"use double quant if you have problems with memory"** → on 6 GB VRAM you **do** have memory pressure → turn it on.

### 4. `bnb_4bit_compute_dtype=torch.bfloat16`
- **What it does**: Controls the **data type used during matrix multiplications** (dequantize → multiply → requantize happens on-the-fly).
  - The **weights stay 4-bit** in memory.
  - But when doing actual computation (forward/backward), bitsandbytes **dequantizes to this dtype** before matmul.


---

In [55]:
# load in 8-bit version for comparison
model_8b = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_use_double_quant=True, bnb_8bit_compute_dtype=torch.bfloat16),
    device_map='cuda:0',
    low_cpu_mem_usage=True,      # Helps reduce peak during load
    trust_remote_code=True
)

Loading weights: 100%|██████████| 388/388 [00:01<00:00, 266.46it/s]


In [56]:
model_8b.model.decoder.project_out.weight.data, model_8b.model.decoder.project_out.weight.shape

(tensor([[  19,   52,  -32,  ...,  -30, -126,   57],
         [  76,  123,  -94,  ...,  -65,  -70,  -64],
         [  88,  -92,  126,  ...,   37,  -64, -123],
         ...,
         [ 126,  -34,  113,  ...,  -18, -126,   67],
         [ -48,   92,  -36,  ...,    9, -125,  -17],
         [  73,  120,  103,  ...,   16,   33,  -86]], device='cuda:0',
        dtype=torch.int8),
 torch.Size([512, 1024]))

### why 8bit tensor is not flattened out?

**4-bit:**
- 2 values per byte → 524,288 values → 262,144 bytes
- To store 262,144 bytes, the shape becomes `[262144, 1]` (heavily flattened)

**8-bit:**
- 1 value per byte → 524,288 values → 524,288 bytes
- To store 524,288 bytes, the shape can stay `[512, 1024]` (no reshaping needed!)
  - Because 512 × 1024 = 524,288 ✓

So 8-bit **doesn't** need to flatten—there's no compression ratio that forces a reshape. Each 8-bit integer can occupy one byte, so the matrix can keep its original 2D structure.

**The key difference:**
- **4-bit packing is aggressive** → multiple values per byte → can't represent as 2D cleanly → gets flattened to 1D
- **8-bit packing is byte-aligned** → one value per byte → maintains 2D structure naturally → no dramatic reshape

Both are still quantized internally (using scales/zeropoints per block), but 8-bit doesn't need the aggressive reshaping because the math works out differently.